In [5]:
from pathlib import Path
import pandas as pd
import joblib

In [6]:
from pathlib import Path

BASE_DIR = Path.cwd().parent

FEATURE_DATA = BASE_DIR / "data" / "processed" / "AQI_feature_engineered.csv"
FEATURE_SCHEMA = BASE_DIR / "models" / "feature_columns.pkl"

FEATURE_STORE_DIR = BASE_DIR / "feature_store"
FEATURE_STORE_FILE = FEATURE_STORE_DIR / "aqi_features.csv"
SCHEMA_FILE = FEATURE_STORE_DIR / "feature_schema.csv"

print("Project root:", BASE_DIR)
print("Feature data:", FEATURE_DATA)
print("Feature schema:", FEATURE_SCHEMA)

Project root: c:\Users\chanchal_soni\Desktop\10_pearlsshine_internship\1st_project\pearls-aqi-predictor
Feature data: c:\Users\chanchal_soni\Desktop\10_pearlsshine_internship\1st_project\pearls-aqi-predictor\data\processed\AQI_feature_engineered.csv
Feature schema: c:\Users\chanchal_soni\Desktop\10_pearlsshine_internship\1st_project\pearls-aqi-predictor\models\feature_columns.pkl


In [7]:
if not FEATURE_DATA.exists():
    raise FileNotFoundError(f"Feature dataset not found: {FEATURE_DATA}")

if not FEATURE_SCHEMA.exists():
    raise FileNotFoundError(f"Feature schema not found: {FEATURE_SCHEMA}")

df = pd.read_csv(FEATURE_DATA)
feature_columns = joblib.load(FEATURE_SCHEMA)

print("Feature dataset shape:", df.shape)
print("Model feature count:", len(feature_columns))

print("\nFeature columns:")
print(feature_columns)

Feature dataset shape: (123, 17)
Model feature count: 16

Feature columns:
['CO', 'NO', 'NO2', 'O3', 'SO2', 'PM2_5', 'PM10', 'NH3', 'hour', 'day', 'month', 'weekday', 'AQI', 'AQI_lag_1', 'AQI_change', 'AQI_rolling_avg']


In [8]:
missing_features = [
    column for column in feature_columns
    if column not in df.columns
]

if missing_features:
    raise ValueError(
        f"Required model features are missing: {missing_features}"
    )

print("All required model features are present.")

All required model features are present.


In [9]:
FEATURE_STORE_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(FEATURE_STORE_FILE, index=False)

print("Feature Store created successfully.")
print("Saved at:", FEATURE_STORE_FILE)

Feature Store created successfully.
Saved at: c:\Users\chanchal_soni\Desktop\10_pearlsshine_internship\1st_project\pearls-aqi-predictor\feature_store\aqi_features.csv


In [10]:
schema_df = pd.DataFrame({
    "feature_name": feature_columns,
    "feature_type": [
        str(df[column].dtype)
        for column in feature_columns
    ]
})

schema_df.to_csv(SCHEMA_FILE, index=False)

print("Feature schema created successfully.")
print("Saved at:", SCHEMA_FILE)

print("\nSchema:")
print(schema_df)

Feature schema created successfully.
Saved at: c:\Users\chanchal_soni\Desktop\10_pearlsshine_internship\1st_project\pearls-aqi-predictor\feature_store\feature_schema.csv

Schema:
       feature_name feature_type
0                CO      float64
1                NO      float64
2               NO2      float64
3                O3      float64
4               SO2      float64
5             PM2_5      float64
6              PM10      float64
7               NH3      float64
8              hour        int64
9               day        int64
10            month        int64
11          weekday        int64
12              AQI        int64
13        AQI_lag_1      float64
14       AQI_change      float64
15  AQI_rolling_avg      float64


In [11]:
feature_store_df = pd.read_csv(FEATURE_STORE_FILE)
schema_check_df = pd.read_csv(SCHEMA_FILE)

assert len(feature_store_df) == len(df)

assert list(schema_check_df["feature_name"]) == list(feature_columns)

assert all(
    column in feature_store_df.columns
    for column in feature_columns
)

print("=" * 55)
print("FEATURE STORE VERIFICATION")
print("=" * 55)

print("Stored rows    :", len(feature_store_df))
print("Stored columns :", len(feature_store_df.columns))
print("Model features :", len(feature_columns))

print("\nFeature Store verification PASSED.")

FEATURE STORE VERIFICATION
Stored rows    : 123
Stored columns : 17
Model features : 16

Feature Store verification PASSED.


In [12]:
print("Latest feature record:")
display(feature_store_df.tail(1))

Latest feature record:


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3,hour,day,month,weekday,AQI_lag_1,AQI_change,AQI_rolling_avg
122,2026-09-04 18:04:48,1,71.29,0.0,1.36,34.41,0.99,6.22,15.86,0.0,18,4,9,4,1.0,0.0,1.0
